# 02 Pipeline

Build a governed pipeline in seven steps: **Environment → Data Contracts → Read → Transform → Target → Validate → Write**.

## Tested with FabricOps

This notebook template is maintained separately from FabricOps package releases. The table below records the FabricOps releases that have been manually tested with this template in Microsoft Fabric.

| FabricOps release | Tested by | Date tested |
|---|---|---|
| v0.2.0 | Voyce | 6 Aug 2026 |

This redesigned version has local structural and public-API compatibility validation only. Run it in your configured Fabric workspace before recording a new runtime test.

# 0. Environment

Run the shared Fabric configuration and import the public APIs used by this pipeline.

In [ ]:
%run 00_env_config

In [ ]:
from pyspark.sql import functions as F

from fabricops_kit import (
    # FabricOps v0.2.0 onwards
    widget_view_catalogue,
    check_dq,
    check_freshness,
    check_schema,
    check_sensitive_data,
    check_source_stability,
    observe_table,
    pipeline_read,
    pipeline_write,
    profile_table,
    resolve_table_id,
    widget_select_data_contract,
)

# 1. Data Contracts

Select the Data Contracts to test with this pipeline. Production automatically uses activated Data Contracts.

In [ ]:
CONTRACTS = widget_select_data_contract()

# 2. Read

Each source uses the same cloneable block. Change only the five `READ_*` values. The block reads the governed source, runs source schema and Data Quality checks, profiles it, and stores the approved DataFrame plus its canonical identity in `sources`. Sensitive Data treatment remains at the governed Write boundary. Optional catalogue inspection is commented out for normal ETL runs.

In [ ]:
sources = {}

## READ 1 — Orders

In [ ]:
READ_NAME = "orders"
READ_STORE = "source"
READ_SCHEMA = "demo"
READ_TABLE = "orders"
READ_QUERY = None

read_result = pipeline_read(
    store=READ_STORE,
    schema=READ_SCHEMA,
    table_name=READ_TABLE,
    query=READ_QUERY,
)

df = read_result["dataframe"]
table_id = read_result["table_id"]

check_schema(table_id, dataframe=df, raise_on_failure=True)
check_dq(df, table_id=table_id, raise_on_failure=True)

profile = profile_table(table_id=table_id)
display(profile["profile"])

sources[READ_NAME] = {
    "dataframe": df,
    "table_id": table_id,
    "has_contract": read_result["has_contract"],
}

# Optional: inspect this source in the current pipeline catalogue.
# catalogue_widget = widget_view_catalogue(mode="pipeline")
# catalogue_widget["show"](table_id=table_id)

## READ 2 — Products

In [ ]:
READ_NAME = "products"
READ_STORE = "source"
READ_SCHEMA = "demo"
READ_TABLE = "products"
READ_QUERY = None

read_result = pipeline_read(
    store=READ_STORE,
    schema=READ_SCHEMA,
    table_name=READ_TABLE,
    query=READ_QUERY,
)

df = read_result["dataframe"]
table_id = read_result["table_id"]

check_schema(table_id, dataframe=df, raise_on_failure=True)
check_dq(df, table_id=table_id, raise_on_failure=True)

profile = profile_table(table_id=table_id)
display(profile["profile"])

sources[READ_NAME] = {
    "dataframe": df,
    "table_id": table_id,
    "has_contract": read_result["has_contract"],
}

# Optional: inspect this source in the current pipeline catalogue.
# catalogue_widget = widget_view_catalogue(mode="pipeline")
# catalogue_widget["show"](table_id=table_id)

## READ 3 — Order History

In [ ]:
READ_NAME = "history"
READ_STORE = "product"
READ_SCHEMA = "demo"
READ_TABLE = "order_history"
READ_QUERY = None

read_result = pipeline_read(
    store=READ_STORE,
    schema=READ_SCHEMA,
    table_name=READ_TABLE,
    query=READ_QUERY,
)

df = read_result["dataframe"]
table_id = read_result["table_id"]

check_schema(table_id, dataframe=df, raise_on_failure=True)
check_dq(df, table_id=table_id, raise_on_failure=True)

profile = profile_table(table_id=table_id)
display(profile["profile"])

sources[READ_NAME] = {
    "dataframe": df,
    "table_id": table_id,
    "has_contract": read_result["has_contract"],
}

# Optional: inspect this source in the current pipeline catalogue.
# catalogue_widget = widget_view_catalogue(mode="pipeline")
# catalogue_widget["show"](table_id=table_id)

# 3. Transform

Business transformations remain project-owned PySpark. Pull the approved source DataFrames from `sources` and transform them normally.

In [ ]:
orders_df = sources["orders"]["dataframe"]
products_df = sources["products"]["dataframe"]
history_df = sources["history"]["dataframe"]

history_summary_df = (
    history_df
    .groupBy("customer_id")
    .agg(
        F.count("*").alias("historical_order_count"),
        F.sum("net_amount").alias("historical_net_amount"),
        F.max("order_datetime").alias("latest_historical_order_datetime"),
    )
)

transformed_df = (
    orders_df.alias("orders")
    .join(products_df.alias("products"), on="product_id", how="left")
    .join(history_summary_df.alias("history"), on="customer_id", how="left")
    .withColumn(
        "order_net_amount",
        F.round(F.col("quantity") * F.col("unit_price") * (F.lit(1.0) - F.col("discount")), 2),
    )
    .fillna({"historical_order_count": 0, "historical_net_amount": 0.0})
    .select(
        "order_id", "customer_id", "order_datetime", "modified_datetime",
        "product_id", "product_name", "product_category", "quantity", "unit_price",
        "discount", "order_net_amount", "order_status", "shipping_country",
        "historical_order_count", "historical_net_amount", "latest_historical_order_datetime",
    )
)
display(transformed_df)

# 4. Target

Describe the governed target and the processing strategy being proposed for this pipeline.

In [ ]:
WRITE_STORE = "unified"
WRITE_SCHEMA = "demo"
WRITE_TABLE = "curated_orders"
WRITE_LOAD_STRATEGY = "overwrite"

# Temporary until target identity is resolved inside the governed Write orchestration.
WRITE_TABLE_ID = resolve_table_id(
    store=WRITE_STORE,
    schema=WRITE_SCHEMA,
    table_name=WRITE_TABLE,
)

# 5. Validate

Complete target-dependent source checks, then run target Guardrails before publication.

## Source validation

Freshness and Source Stability currently require the source-to-target relationship, so they run here after the target is known.

In [ ]:
for source in sources.values():
    if not source["has_contract"]:
        continue

    observation = observe_table(
        table_id=source["table_id"],
        target_table_id=WRITE_TABLE_ID,
    )
    check_freshness(observation, raise_on_failure=True)
    check_source_stability(
        observation,
        target_table_id=WRITE_TABLE_ID,
    )

In [ ]:
check_schema(WRITE_TABLE_ID, dataframe=transformed_df, raise_on_failure=True)
check_dq(transformed_df, table_id=WRITE_TABLE_ID, raise_on_failure=True)

sensitive_result = check_sensitive_data(
    transformed_df,
    table_id=WRITE_TABLE_ID,
)
if not sensitive_result["can_continue"]:
    raise RuntimeError("A blocking Sensitive Data Guardrail failed for the target.")

prepared_df = sensitive_result["dataframe"]

# 6. Write

Publish through `pipeline_write()`, then profile the persisted target. FabricOps commits Lineage and successful Source Observation state only after publication succeeds. Optional catalogue inspection is commented out for normal ETL runs.

## WRITE 1 — Curated Orders

In [ ]:
write_result = pipeline_write(
    prepared_df,
    store=WRITE_STORE,
    schema=WRITE_SCHEMA,
    table_name=WRITE_TABLE,
    load_strategy=WRITE_LOAD_STRATEGY,
    source_table_ids=[source["table_id"] for source in sources.values()],
)

write_profile = profile_table(table_id=write_result["table_id"])
display(write_profile["profile"])

# Optional: inspect the persisted target in the current pipeline catalogue.
# catalogue_widget = widget_view_catalogue(mode="pipeline")
# catalogue_widget["show"](table_id=write_result["table_id"])